<a href="https://colab.research.google.com/github/Matheus-200/python_sis_26_2/blob/Matheus_Gabriel/aula_11/matheus_gabriel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
### Etapa 1: Importações e Padrão PEP 8
```

In [1]:
# Módulos nativos
import glob
import os
import xml.etree.ElementTree as ET

# Módulos de terceiros
import pandas as pd

```markdown
### Etapa 2: Coleta e Varredura de XMLs (Criação das Amostras)
```

In [8]:
import os

os.makedirs("notas_xml", exist_ok=True)

notas_amostra = [
    {"num": "1001", "data": "2026-10-10", "fornecedor": "Tech Solutions LTDA", "cnpj": "12.345.678/0001-90", "total": "4500.00"},
    {"num": "1002", "data": "2026-10-12", "fornecedor": "Cloud Cloud Servicos", "cnpj": "98.765.432/0001-11", "total": "1280.50"},
    {"num": "1003", "data": "2026-10-15", "fornecedor": "Hardware & Redes SA", "cnpj": "45.123.890/0001-55", "total": "8900.00"}
]

for n in notas_amostra:
    conteudo = f"""<nfeProc xmlns="http://www.portalfiscal.inf.br/nfe">
    <NFe>
        <infNFe>
            <ide>
                <numero>{n['num']}</numero>
                <data_emissao>{n['data']}T14:30:00-03:00</data_emissao>
            </ide>
            <emit>
                <cnpj>{n['cnpj']}</cnpj>
                <razao_social>{n['fornecedor']}</razao_social>
            </emit>
            <total>
                <valor_total>{n['total']}</valor_total>
            </total>
        </infNFe>
    </NFe>
</nfeProc>
"""
    with open(f"notas_xml/nfe_{n['num']}.xml", "w", encoding="utf-8") as f:
        f.write(conteudo)

print("Amostras criadas na pasta 'notas_xml' com sucesso!")

Amostras criadas na pasta 'notas_xml' com sucesso!


```markdown
### Etapa 3: Extração
```

In [9]:
dados_notas = []

caminho_xmls = glob.glob("notas_xml/*.xml")

for caminho in caminho_xmls:
    try:
        tree = ET.parse(caminho)
        root = tree.getroot()

        # O XML fornecido possui namespaces que podemos ignorar ou buscar diretamente por tags
        # Como a estrutura simulada é simples, buscamos de forma genérica:
        numero = root.find(".//numero").text
        data = root.find(".//data_emissao").text[:10]  # Obtendo apenas a data YYYY-MM-DD
        fornecedor = root.find(".//razao_social").text
        cnpj = root.find(".//cnpj").text
        valor_total = float(root.find(".//valor_total").text)

        dados_notas.append({
            "Número da Nota": numero,
            "Data de Emissão": data,
            "Fornecedor": fornecedor,
            "CNPJ": cnpj,
            "Valor Total": valor_total
        })
    except Exception as e:
        print(f"Erro ao processar o arquivo {caminho}: {e}")

print(f"Sucesso! {len(dados_notas)} notas fiscais foram processadas.")

Erro ao processar o arquivo notas_xml/nfe_1002.xml: 'NoneType' object has no attribute 'text'
Erro ao processar o arquivo notas_xml/nfe_1001.xml: 'NoneType' object has no attribute 'text'
Erro ao processar o arquivo notas_xml/nfe_1003.xml: not well-formed (invalid token): line 10, column 40
Sucesso! 0 notas fiscais foram processadas.


```markdown
### Etapa 4: Geração da Planilha
```

In [10]:
# Criar DataFrame
df_notas = pd.DataFrame(dados_notas)

# Verificar se existem dados para evitar KeyError
if not df_notas.empty:
    # Formatando o valor monetário com duas casas decimais
    df_notas["Valor Total"] = df_notas["Valor Total"].map("{:.2f}".format)

    nome_arquivo = "auditoria_notas.xlsx"
    df_notas.to_excel(nome_arquivo, index=False)
    print(f"Planilha '{nome_arquivo}' gerada com sucesso!")
    display(df_notas)
else:
    print("Nenhum dado encontrado para gerar a planilha. Certifique-se de executar o script da Etapa 2 para criar as notas simuladas primeiro!")

Nenhum dado encontrado para gerar a planilha. Certifique-se de executar o script da Etapa 2 para criar as notas simuladas primeiro!
